In [8]:
import os
import sys

PYTHON_PATH = r"C:\Users\Shubham\AppData\Local\Programs\Python\Python310\python.exe"
# Force Spark to use same Python for driver & workers
os.environ["PYSPARK_PYTHON"] = PYTHON_PATH
os.environ["PYSPARK_DRIVER_PYTHON"] = PYTHON_PATH

print("Python executable:", sys.executable)
print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])


Python executable: C:\Users\Shubham\AppData\Local\Programs\Python\Python310\python.exe
PYSPARK_PYTHON: C:\Users\Shubham\AppData\Local\Programs\Python\Python310\python.exe


In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

spark = SparkSession.builder \
    .appName("FoodNutritionAnalyzerBalanced") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("Spark version:", spark.version)


Spark version: 3.5.1


In [11]:
file_path = r"D:\CDAC PROJECT\FoodNutritionAnalyzer\data\en.openfoodfacts.org.products.csv"

selected_cols = [
    "product_name",
    "nutriscore_grade",
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates-total_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "sodium_100g",
    "additives_n",
    "nova_group"
]

df = spark.read \
    .option("header", "true") \
    .option("sep", "\t") \
    .option("quote", "\u0000") \
    .option("mode", "DROPMALFORMED") \
    .csv(file_path) \
    .select(*selected_cols)

print("Rows loaded:", df.count())


Rows loaded: 4285540


In [14]:
numeric_cols = [
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates-total_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "sodium_100g",
    "additives_n",
    "nova_group"
]

for c in numeric_cols:
    df = df.withColumn(c, col(c).cast("double"))


In [15]:
df = df.filter(col("nutriscore_grade").isin("a", "b", "c", "d", "e"))

df = df.withColumn(
    "health_label",
    when(col("nutriscore_grade").isin("a", "b"), 1).otherwise(0)
)

df.groupBy("health_label").count().show()


+------------+------+
|health_label| count|
+------------+------+
|           1|347958|
|           0|993357|
+------------+------+



In [16]:
healthy_df = df.filter(col("health_label") == 1)
unhealthy_df = df.filter(col("health_label") == 0)

healthy_count = healthy_df.count()
unhealthy_count = unhealthy_df.count()

sampling_ratio = healthy_count / unhealthy_count

unhealthy_sampled = unhealthy_df.sample(
    withReplacement=False,
    fraction=sampling_ratio,
    seed=42
)

balanced_df = healthy_df.union(unhealthy_sampled)

balanced_df.groupBy("health_label").count().show()


+------------+------+
|health_label| count|
+------------+------+
|           1|347958|
|           0|347738|
+------------+------+



In [17]:
balanced_df = balanced_df.fillna(0)


In [18]:
feature_cols = [
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates-total_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "sodium_100g",
    "additives_n",
    "nova_group"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

final_df = assembler.transform(balanced_df).select(
    "features", "health_label"
)


In [19]:
train_df, test_df = final_df.randomSplit([0.8, 0.2], seed=42)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())


Train rows: 556602
Test rows: 139094


In [20]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="health_label",
    maxIter=50
)

model = lr.fit(train_df)
print("✅ Model trained on balanced data")


✅ Model trained on balanced data


In [21]:
predictions = model.transform(test_df)

evaluator = BinaryClassificationEvaluator(
    labelCol="health_label",
    metricName="areaUnderROC"
)


auc = evaluator.evaluate(predictions)
print("✅ AUC (Balanced Data):", auc)


✅ AUC (Balanced Data): 0.935710881639125


In [23]:
from pyspark.sql.functions import col

cm_df = predictions.select(
    col("health_label").cast("int").alias("label"),
    col("prediction").cast("int").alias("prediction")
)

cm_df.groupBy("label", "prediction").count().show()


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|         0| 6150|
|    1|         1|63292|
|    0|         1|12591|
|    0|         0|57061|
+-----+----------+-----+



In [24]:
# Check schema
predictions.printSchema()

# Show some predictions
predictions.select(
    "health_label",
    "prediction",
    "probability"
).show(10, truncate=False)


root
 |-- features: vector (nullable = true)
 |-- health_label: integer (nullable = false)
 |-- rawPrediction: vector (nullable = true)
 |-- probability: vector (nullable = true)
 |-- prediction: double (nullable = false)

+------------+----------+----------------------------------------+
|health_label|prediction|probability                             |
+------------+----------+----------------------------------------+
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.022393394697

In [25]:
from pyspark.sql.functions import col

confusion_df = predictions.groupBy(
    col("health_label").alias("label"),
    col("prediction")
).count()

confusion_df.show()


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       1.0|63292|
|    1|       0.0| 6150|
|    0|       0.0|57061|
|    0|       1.0|12591|
+-----+----------+-----+



In [26]:
# Extract counts
tp = predictions.filter("health_label = 1 AND prediction = 1").count()
tn = predictions.filter("health_label = 0 AND prediction = 0").count()
fp = predictions.filter("health_label = 0 AND prediction = 1").count()
fn = predictions.filter("health_label = 1 AND prediction = 0").count()

total = tp + tn + fp + fn

accuracy  = (tp + tn) / total
precision = tp / (tp + fp)
recall    = tp / (tp + fn)
f1_score  = 2 * (precision * recall) / (precision + recall)

print("✅ MODEL TEST RESULTS")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1_score:.4f}")


✅ MODEL TEST RESULTS
Accuracy : 0.8653
Precision: 0.8341
Recall   : 0.9114
F1 Score : 0.8710


In [27]:
# Extract coefficients from Spark Logistic Regression model
import numpy as np
coef = np.array(model.coefficients.toArray())
intercept = model.intercept

print("✅ Coefficients loaded")
print("Coef length:", len(coef))
print("Intercept:", intercept)


✅ Coefficients loaded
Coef length: 11
Intercept: 3.776341309234545


In [28]:
# Feature order MUST match training exactly
features = [
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates-total_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "sodium_100g",
    "additives_n",
    "nova_group"
]

print("✅ Features loaded:", features)


✅ Features loaded: ['energy-kcal_100g', 'fat_100g', 'saturated-fat_100g', 'carbohydrates-total_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g', 'additives_n', 'nova_group']


In [29]:
import numpy as np

def predict_food_health(food_dict):
    """
    Predict health label and confidence using trained Logistic Regression model
    food_dict: nutrition values per 100g
    """

    # Ensure feature order matches training
    X = np.array([food_dict[f] for f in features])

    # Logistic regression equation
    z = np.dot(coef, X) + intercept
    prob_healthy = 1 / (1 + np.exp(-z))

    label = "Healthy" if prob_healthy >= 0.5 else "Unhealthy"
    return label, prob_healthy


In [30]:
sample_food_healthy = {
    "energy-kcal_100g": 120,
    "fat_100g": 3,
    "saturated-fat_100g": 0.5,
    "carbohydrates-total_100g": 18,
    "sugars_100g": 2,
    "fiber_100g": 8,
    "proteins_100g": 10,
    "salt_100g": 0.05,
    "sodium_100g": 0.02,
    "additives_n": 0,
    "nova_group": 1
}

label, confidence = predict_food_health(sample_food_healthy)

print("🥗 HEALTHY SAMPLE TEST")
print("Prediction:", label)
print(f"Healthy confidence: {confidence*100:.2f}%")


🥗 HEALTHY SAMPLE TEST
Prediction: Healthy
Healthy confidence: 92.47%


In [31]:
from pyspark.sql.functions import col

# predictions = model.transform(test_df)

confusion_df = predictions.groupBy(
    col("health_label").alias("Actual"),
    col("prediction").alias("Predicted")
).count()

confusion_df.show()


+------+---------+-----+
|Actual|Predicted|count|
+------+---------+-----+
|     1|      1.0|63292|
|     1|      0.0| 6150|
|     0|      0.0|57061|
|     0|      1.0|12591|
+------+---------+-----+



In [32]:
#ocr

In [33]:
import cv2
import pytesseract
import re
import numpy as np


In [34]:
def extract_text_from_image(image_path):
    img = cv2.imread(image_path)

    if img is None:
        raise ValueError("Invalid image path")

    img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.bilateralFilter(gray, 11, 17, 17)

    thresh = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31, 2
    )

    config = r'--oem 3 --psm 6'
    return pytesseract.image_to_string(thresh, config=config)


In [35]:
def parse_nutrients(text):
    text = text.lower()

    patterns = {
        "energy_100g": r"energy[^\d]*([\d]+\.?[\d]*)",
        "fat_100g": r"(?:total fat|fat)[^\d]*([\d]+\.?[\d]*)",
        "saturated_fat_100g": r"saturated fat[^\d]*([\d]+\.?[\d]*)",
        "carbohydrates_100g": r"carbohydrate[^\d]*([\d]+\.?[\d]*)",
        "sugars_100g": r"(?:total sugars|added sugars|sugars)[^\d]*([\d]+\.?[\d]*)",
        "proteins_100g": r"protein[^\d]*([\d]+\.?[\d]*)",
        "salt_100g": r"salt[^\d]*([\d]+\.?[\d]*)",
        "sodium_100g": r"sodium[^\d]*([\d]+\.?[\d]*)"
    }

    nutrients = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, text)
        nutrients[key] = float(match.group(1)) if match else 0.0

    # Sodium → salt fallback
    if nutrients["salt_100g"] == 0 and nutrients["sodium_100g"] > 0:
        nutrients["salt_100g"] = (nutrients["sodium_100g"] / 1000) * 2.5

    return nutrients


In [36]:
def infer_additives_n(text):
    additive_keywords = [
        "emulsifier", "stabilizer", "preservative",
        "flavour", "flavor", "colour", "color",
        "sweetener", "enzyme", "thickener"
    ]

    count = sum(1 for w in additive_keywords if w in text)
    if "ingredients" in text and count == 0:
        return 1
    return count


def infer_nova_group(text):
    ultra = [
        "flavour", "emulsifier", "stabilizer", "sweetener",
        "enzyme", "isolated", "protein powder", "instant",
        "fortified", "artificial"
    ]
    processed = ["cheese", "bread", "butter", "salted", "roasted"]
    natural = ["fruit", "vegetable", "grain", "milk", "egg", "rice"]

    if any(k in text for k in ultra):
        return 4
    elif any(k in text for k in processed):
        return 3
    elif any(k in text for k in natural):
        return 1
    return 4


In [37]:
FEATURES = [
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates-total_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "sodium_100g",
    "additives_n",
    "nova_group"
]


In [38]:
import numpy as np

def predict_food_health(model, food_dict):
    """
    Predict health label and confidence using trained Spark Logistic Regression
    """

    # Extract coefficients
    coef = model.coefficients.toArray()
    intercept = model.intercept

    # Build feature vector in correct order
    X = np.array([food_dict[f] for f in FEATURES])

    # Logistic regression equation
    z = np.dot(coef, X) + intercept
    prob_healthy = 1 / (1 + np.exp(-z))

    pred = 1 if prob_healthy >= 0.5 else 0
    return pred, [1 - prob_healthy, prob_healthy]


In [1]:
 image_path = r"D:\CDAC PROJECT\FoodNutritionAnalyzer\Screenshot 2026-01-26 152844.png"


text = extract_text_from_image(image_path)
print("📝 OCR TEXT:\n", text)

nutrients = parse_nutrients(text)

nutrients["additives_n"] = infer_additives_n(text)
nutrients["nova_group"] = infer_nova_group(text)

sample_food = {
    "energy-kcal_100g": nutrients["energy_100g"],
    "fat_100g": nutrients["fat_100g"],
    "saturated-fat_100g": nutrients["saturated_fat_100g"],
    "carbohydrates-total_100g": nutrients["carbohydrates_100g"],
    "sugars_100g": nutrients["sugars_100g"],
    "fiber_100g": nutrients.get("fiber_100g", 0.0),
    "proteins_100g": nutrients["proteins_100g"],
    "salt_100g": nutrients["salt_100g"],
    "sodium_100g": nutrients["sodium_100g"] / 1000,
    "additives_n": nutrients["additives_n"],
    "nova_group": nutrients["nova_group"]
}

pred, prob = predict_food_health(model, sample_food)

label = "Healthy" if pred == 1 else "Unhealthy"

print("Prediction:", label)
print("Probability (Unhealthy, Healthy):", prob)


NameError: name 'extract_text_from_image' is not defined

In [43]:
print(sample_food)

{'energy-kcal_100g': 60.0, 'fat_100g': 9.0, 'saturated-fat_100g': 9.0, 'carbohydrates-total_100g': 9.0, 'sugars_100g': 9.0, 'fiber_100g': 0.0, 'proteins_100g': 29.0, 'salt_100g': 0.0, 'sodium_100g': 0.0, 'additives_n': 0, 'nova_group': 4}


In [40]:
# Spark LogisticRegressionModel
spark_model = model   # your trained model

coef = spark_model.coefficients.toArray()
intercept = spark_model.intercept

print("Intercept:", intercept)
print("Coefficients:", coef)


Intercept: 3.776341309234545
Coefficients: [ 3.65891540e-18  3.54729444e-02 -4.81615135e-01 -2.86633329e-02
 -2.78118440e-01 -3.09059221e-19  2.18649064e-07 -1.18014408e+00
 -2.95036974e+00 -2.43663566e-01  5.58510389e-02]


In [41]:
features = [
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates-total_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "sodium_100g",
    "additives_n",
    "nova_group"
]


In [33]:
import pickle
import numpy as np

class FoodHealthModel:
    def __init__(self, coef, intercept, features):
        self.coef = np.array(coef)
        self.intercept = intercept
        self.features = features

    def predict_proba(self, X):
        z = np.dot(X, self.coef) + self.intercept
        prob_healthy = 1 / (1 + np.exp(-z))
        return np.array([[1 - prob_healthy, prob_healthy]])

    def predict(self, X):
        return (self.predict_proba(X)[0][1] >= 0.5).astype(int)


In [34]:
model = FoodHealthModel(
    coef=coef,
    intercept=intercept,
    features=features
)

with open("food_health_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("✅ food_health_model.pkl saved")


✅ food_health_model.pkl saved


In [35]:
print(type(model))


<class '__main__.FoodHealthModel'>
